# MethodThinker 阿里云PAI训练 Notebook

> 全量数据训练 (13,340样本) | QLoRA模式 | 阿里云GPU优化

## 适用平台
- 阿里云PAI-DSW
- 阿里云PAI-EAS
- 阿里云ECS GPU实例

## 绑定目录
- **数据盘**: `/mnt/data` (持久化存储)
- **训练结果**: `/mnt/data/models/{日期}/`

## 训练配置
- **模型**: Qwen/Qwen2.5-Math-1.5B
- **数据**: 13,340样本
- **模式**: QLoRA (4-bit量化 + LoRA)
- **预计时间**: 2-4小时 (V100/A10)

## 1. 环境检查

In [ ]:
# 检查GPU
import torch
print("="*50)
print("阿里云GPU环境检查")
print("="*50)
if torch.cuda.is_available():
    print(f"✓ GPU型号: {torch.cuda.get_device_name(0)}")
    print(f"✓ 显存大小: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    print(f"✓ CUDA版本: {torch.version.cuda}")
    print(f"✓ 支持BF16: {torch.cuda.is_bf16_supported()}")
else:
    print("✗ 未检测到GPU!")

In [ ]:
# 设置HuggingFace镜像和下载优化（国内加速）
import os

# 设置镜像
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

# 设置缓存到绑定目录（持久化，避免重复下载）
cache_dir = '/mnt/data/huggingface_cache'
os.makedirs(cache_dir, exist_ok=True)
os.environ['HF_HOME'] = cache_dir
os.environ['HF_HUB_CACHE'] = cache_dir

print("✓ 已设置HuggingFace镜像")
print(f"✓ 缓存目录: {cache_dir}")

## 2. 项目准备

In [ ]:
# 安装依赖
!pip install -q transformers>=4.40.0 accelerate>=0.25.0 peft>=0.7.0 trl>=0.7.0 datasets>=2.14.0 bitsandbytes>=0.41.0

# 安装Flash Attention（可选，大幅加速训练）
# 需要30分钟左右编译，首次安装后缓存
# !pip install flash-attn --no-build-isolation

print("✓ 依赖安装完成")
print("提示: 如需Flash Attention加速，取消上面注释安装（首次需要编译）")

## 3. 预下载模型

自动检测缓存，避免重复下载：

In [ ]:
# 预下载模型（自动跳过已缓存的）
from huggingface_hub import snapshot_download, try_to_load_from_cache
import os

model_name = "Qwen/Qwen2.5-Math-1.5B"
cache_dir = '/mnt/data/huggingface_cache'

print(f"检查模型缓存: {model_name}")

# 检查是否已缓存
try:
    cached_path = try_to_load_from_cache(
        repo_id=model_name,
        filename="config.json",
        cache_dir=cache_dir
    )
    if cached_path:
        print(f"✓ 模型已缓存，跳过下载")
        print(f"  缓存路径: {cached_path}")
    else:
        raise FileNotFoundError("未找到缓存")
except (FileNotFoundError, Exception):
    print(f"模型未缓存，开始下载...")
    print("="*50)
    local_path = snapshot_download(
        repo_id=model_name, 
        cache_dir=cache_dir
    )
    print(f"✓ 模型下载完成: {local_path}")

## 4. 训练参数优化

根据GPU显存自动调整参数：

In [ ]:
# 根据GPU显存选择配置
import torch

gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3

if gpu_memory >= 40:  # A100 40GB+
    config = {
        "batch_size": 16,
        "max_length": 4096,
        "gradient_accumulation": 2,
        "lora_r": 64,
        "samples": 13340
    }
elif gpu_memory >= 24:  # A10/RTX 3090/4090
    config = {
        "batch_size": 8,
        "max_length": 2048,
        "gradient_accumulation": 4,
        "lora_r": 32,
        "samples": 10000
    }
else:  # T4 16GB
    config = {
        "batch_size": 4,
        "max_length": 1024,
        "gradient_accumulation": 8,
        "lora_r": 16,
        "samples": 5000
    }

print("自动选择的训练配置:")
for k, v in config.items():
    print(f"  {k}: {v}")

## 5. 开始训练

In [ ]:
# 创建按日期命名的输出目录
import torch
from datetime import datetime
import os

# 切换到项目根目录
project_root = "/mnt/workspace/method-thinker"
if os.path.exists(project_root):
    os.chdir(project_root)
    print(f"✓ 工作目录: {os.getcwd()}")
else:
    print(f"✗ 项目目录不存在: {project_root}")

# 获取当前日期
today = datetime.now().strftime("%Y-%m-%d")
output_dir = f"/mnt/data/models/{today}"

# 创建目录
os.makedirs(output_dir, exist_ok=True)
print(f"✓ 输出目录: {output_dir}")

# 检测GPU配置 - 启用gradient checkpointing后可增大batch
gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3

if gpu_memory >= 40:
    # A100 40GB+: 大batch全量训练
    !python scripts/train_sft.py \
        --train-data data/train_data/all_merged.json \
        --use-lora \
        --epochs 3 \
        --batch-size 32 \
        --max-length 4096 \
        --output-dir {output_dir}
elif gpu_memory >= 24:
    # A10/RTX 3090/4090: 中等配置
    !python scripts/train_sft.py \
        --train-data data/train_data/all_merged.json \
        --use-lora \
        --epochs 2 \
        --batch-size 16 \
        --max-length 2048 \
        --output-dir {output_dir}
elif gpu_memory >= 16:
    # T4 16GB: 使用gradient checkpointing
    !python scripts/train_sft.py \
        --train-data data/train_data/all_merged.json \
        --use-lora \
        --epochs 1 \
        --batch-size 8 \
        --max-length 1024 \
        --output-dir {output_dir}
else:
    # 低显存
    !python scripts/train_sft.py \
        --train-data data/train_data/all_merged.json \
        --use-lora \
        --epochs 1 \
        --batch-size 4 \
        --max-length 512 \
        --output-dir {output_dir}

print(f"\n✓ 训练完成，模型保存在: {output_dir}")

## 6. 模型保存位置

### 阿里云绑定目录
- **数据盘**: `/mnt/data` (持久化存储)
- **模型保存**: `/mnt/data/models/{日期}/`
- **自动命名**: 按训练日期自动创建目录

In [ ]:
# 检查模型保存结果
import os

data_dir = "/mnt/data"
models_dir = os.path.join(data_dir, "models")

if os.path.exists(models_dir):
    dates = sorted(os.listdir(models_dir))
    print(f"已保存的模型 ({len(dates)}个版本):")
    for date in dates:
        model_path = os.path.join(models_dir, date)
        if os.path.isdir(model_path):
            files = os.listdir(model_path)
            total_size = sum(
                os.path.getsize(os.path.join(model_path, f)) / 1024 / 1024
                for f in files if os.path.isfile(os.path.join(model_path, f))
            )
            print(f"  {date}: {len(files)}文件, {total_size:.1f}MB")
else:
    print("暂无已保存模型")

## 7. 打包下载

In [ ]:
# 打包最新模型
import os

data_dir = "/mnt/data"
models_dir = os.path.join(data_dir, "models")

if os.path.exists(models_dir):
    dates = sorted(os.listdir(models_dir), reverse=True)
    if dates:
        latest_date = dates[0]
        tar_file = f"/mnt/data/method-thinker-{latest_date}.tar.gz"
        !cd /mnt/data/models && tar -czvf {tar_file} {latest_date}/
        
        file_size = os.path.getsize(tar_file) / 1024 / 1024
        print(f"✓ 模型已打包: {tar_file}")
        print(f"  文件大小: {file_size:.1f} MB")
    else:
        print("✗ 未找到已训练模型")
else:
    print("✗ 模型目录不存在")

## 8. 验证模型

In [ ]:
# 测试最新模型
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import os

data_dir = "/mnt/data"
models_dir = os.path.join(data_dir, "models")

if os.path.exists(models_dir):
    dates = sorted(os.listdir(models_dir), reverse=True)
    if dates:
        latest_date = dates[0]
        model_path = os.path.join(models_dir, latest_date)
        
        print(f"加载模型: {model_path}")
        tokenizer = AutoTokenizer.from_pretrained(model_path)
        model = AutoModelForCausalLM.from_pretrained(
            model_path,
            device_map="auto",
            torch_dtype=torch.float16
        )
        
        # 测试推理
        test_input = "解方程: 2x + 3 = 7"
        inputs = tokenizer(test_input, return_tensors="pt").to(model.device)
        outputs = model.generate(**inputs, max_new_tokens=256)
        
        print(f"\n问题: {test_input}")
        print(f"回答: {tokenizer.decode(outputs[0], skip_special_tokens=True)}")
        print(f"\n✓ 模型验证成功")
    else:
        print("✗ 未找到已训练模型")
else:
    print("✗ 模型目录不存在，请先运行训练")